## Data Cleaning, Missing Observations and Feature Engineering 

In [10]:
import pandas as pd
import numpy as np
import json
import re

from math import sqrt, sin, cos, asin, radians
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# --- Data Loading ---
def load_dataset(path: str) -> pd.DataFrame:
    return pd.read_csv(path)
# Load dataset 
df_train = pd.read_csv('dataset/train.csv') 
df_test = pd.read_csv('dataset/test.csv') 
df = pd.concat([df_test, df_train])

### 1. Numerical Feature and Target Variable Cleaning
This step converts all numerical features and the target variable (`price`) into a consistent numeric format suitable for machine learning models. Several variables are stored as object types due to the presence of symbols or text, which must be removed before training.

- **Percentage-based features**: Percentage symbols are removed from `host_response_rate` and `host_acceptance_rate`, and values are converted to numeric format.
- **Monetary values**: Currency symbols and thousand separators are removed from the `price` column to extract numerical values.
- **Mixed text–numeric features**: Textual representations in the `bathrooms` feature (e.g. “half”) are converted to numeric equivalents and numeric values are extracted.
- **Data type standardisation**: All cleaned numerical features are converted to floating-point format to ensure consistency and compatibility with regression algorithms.

This process ensures that all numerical inputs are clean, standardised, and ready for downstream modelling.


In [11]:
# --- Numerical Feature and Target Variable Cleaning ---
def clean_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean numerical features and the target variable by removing text and symbols,
    converting all values into a consistent numeric format suitable for modelling.
    """
    # Create a copy to avoid in-place modification
    df = df.copy()

    # Percentage-based features
    percent_cols = ['host_response_rate', 'host_acceptance_rate']
    for col in percent_cols:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = (
                df[col]
                .str.replace('%', '', regex=False)
                .astype(float)
                / 100
            )

    # Monetary values (target variable included)
    if 'price' in df.columns and df['price'].dtype == 'object':
        df['price'] = (
            df['price']
            .str.replace('[\$,]', '', regex=True)
            .astype(float)
        )

    # Mixed text–numeric feature: bathrooms
    if 'bathrooms' in df.columns and df['bathrooms'].dtype == 'object':
        df['bathrooms'] = (
            df['bathrooms']
            .str.replace('half', '0.5', case=False, regex=False)
            .str.extract(r'(\d+\.?\d*)')[0]
            .astype(float)
        )

    return df

### 2. Feature Engineering from Multi-Value Attributes
Some features in the dataset contain multiple pieces of information stored within a single column, limiting their direct usefulness for predictive modelling. In this step, multi-value attributes are decomposed into structured numerical features to better capture host credibility and property quality.

Two columns are selected for transformation: `host_verifications` and `amenities`.

- **Feature 1: Host verification features**:  
  The `host_verifications` column is processed to quantify host credibility. A new feature is created to represent the total number of verification methods completed by the host, providing a continuous measure of trustworthiness rather than a raw text list.

- **Feature 2: Amenity availability features**:  
  The `amenities` column is transformed into multiple informative variables. First, a total amenity count is computed to summarise the overall level of property provisioning. This captures the general completeness of the listing.

- **Feature 3-9: Luxury amenity indicators**:  
7 binary features are created to capture the presence of high-value amenities commonly associated with premium listings:
  - `has_pool`: Indicates availability of a swimming pool or hot tub.
  - `has_heating_cooling`: Indicates presence of air conditioning and/or heating systems.
  - `has_coffee_maker`: Indicates availability of coffee or espresso-making equipment.
  - `has_parking`: Indicates availability of on-site or street parking.
  - `has_private_balcony`: Indicates presence of a private balcony, patio, or outdoor space.
  - `has_high_quality_bedding`: Indicates availability of premium bedding or extra linens.
  - `has_scenic_view`: Indicates listings advertised with views (e.g. city, garden, or waterfront views).

- **Feature 10: Composite luxury score**:  
  To provide a compact representation of property quality, all luxury amenity indicators are aggregated into a single composite feature. This summarised score reflects the overall level of premium amenities offered by each listing.

By decomposing multi-value attributes into count-based, binary, and composite features, this step improves feature interpretability and enables the model to learn both granular and high-level patterns related to listing quality and pricing.

In [14]:
def create_new_features(df):
    """
    Create new features from existing multi-information columns (host_verifications and amenities).
    Extracts counts and key indicators to convert complex text data into usable numerical features.
    """    # -----------------------------
    # Feature 1: Host verification count
    # -----------------------------
    if 'host_verifications' in df.columns:
        df['verification_count'] = (
            df['host_verifications']
            .fillna('')
            .apply(lambda x: len(x.split(',')) if x.strip() != '' else 0)
        )

    # -----------------------------
    # Feature 2: Total amenity count
    # -----------------------------
    if 'amenities' in df.columns:
        df['amenity_count'] = (
            df['amenities']
            .fillna('')
            .apply(lambda x: len(x.split(',')) if x.strip() != '' else 0)
        )

        # -----------------------------
        # Feature 3–9: Luxury amenity indicators
        # -----------------------------
        luxury_amenities = {
            'has_pool': ['pool', 'hot tub'],
            'has_heating_cooling': ['air conditioning', 'ac', 'heating', 'central heating'],
            'has_coffee_maker': ['coffee maker', 'nespresso', 'espresso'],
            'has_parking': ['free parking', 'paid parking', 'garage', 'carport', 'street parking'],
            'has_private_balcony': ['private patio or balcony', 'balcony', 'terrace'],
            'has_high_quality_bedding': [
                'high quality linens', 'premium linens',
                'extra pillows and blankets', 'blackout curtains',
                'room-darkening shades', 'comfortable bedding'
            ],
            'has_scenic_view': ['view']
        }

        for feature, keywords in luxury_amenities.items():
            pattern = r'|'.join(rf'\b{re.escape(k)}\b' for k in keywords)
            df[feature] = (
                df['amenities']
                .fillna('')
                .str.contains(pattern, case=False, regex=True)
                .astype(int)
            )

        # -----------------------------
        # Feature 10: Composite luxury score
        # -----------------------------
        luxury_cols = list(luxury_amenities.keys())
        df['luxury_amenity_count'] = df[luxury_cols].sum(axis=1)

    return df

### 3. Missing Value Imputation 

### 4. Categorical Feature Grouping and Encoding

### 5. Additional Data Transformations Prior to Modelling

### 6. Additional Data Transformations Prior to Modelling